In [ ]:
import os
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader
import gym
import d4rl  # noqa: F401

from src.experiment_config import *
from src.config import CHECKPOINTS_DIR, RAW_METRICS_DIR, OBS_STATS_DIR
from src.dataset import NoisyOfflineRLDataset
from src.encoder import DisentangledEncoder
from src.iql import IQLAgent
from src.train_eval import (
    eval_policy_on_env,
    load_and_freeze_encoder,
    train_iql_from_loader,
    save_metrics_json,
)

In [ ]:
def rbf_kernel(x, sigma=1.0):
    dist = torch.cdist(x, x, p=2.0) ** 2
    return torch.exp(-dist / (2.0 * (sigma ** 2)))

def _center_kernel(K):
    row_mean = K.mean(dim=1, keepdim=True)
    col_mean = K.mean(dim=0, keepdim=True)
    total_mean = K.mean()
    return K - row_mean - col_mean + total_mean

def hsic_loss(z1, z2, sigma=1.0):
    n = z1.size(0)
    K = rbf_kernel(z1, sigma=sigma)
    L = rbf_kernel(z2, sigma=sigma)
    Kc = _center_kernel(K)
    Lc = _center_kernel(L)
    return (Kc * Lc).sum() / ((n - 1) ** 2)

In [ ]:
def barlow_cross_correlation_loss(z1, z2):
    z1_norm = (z1 - z1.mean(dim=0)) / (z1.std(dim=0) + 1e-5)
    z2_norm = (z2 - z2.mean(dim=0)) / (z2.std(dim=0) + 1e-5)
    batch_size = z1.size(0)
    c = torch.mm(z1_norm.T, z2_norm) / batch_size
    loss = torch.sum(c ** 2)
    return loss

In [ ]:
# Fixed experiment settings for lambda sensitivity sweep.
# Change ENV_NAME and SEED via environment variables or edit below.
ENV_NAME = os.environ.get("ENV_NAME", "halfcheetah-medium-v2")
SEED = int(os.environ.get("SEED", 1))
NOISE_DIM = int(os.environ.get("NOISE_DIM", 17))
NOISE_SCALE = float(os.environ.get("NOISE_SCALE", 1.0))
NOISE_TYPE = os.environ.get("NOISE_TYPE", "nonlinear")

BATCH_SIZE = 256
PRETRAIN_BS = 512
PRETRAIN_EPOCHS = 20
EPOCHS = 100
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Lambda values to sweep.
LAMBDA_VALUES = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0, 500.0]
# Methods to sweep: "hsic" and "barlow"
SWEEP_METHODS = ["hsic", "barlow"]

from src.config import CHECKPOINTS_DIR, RAW_METRICS_DIR, OBS_STATS_DIR

def noise_tag(noise_dim, noise_scale, noise_type):
    ns = str(noise_scale).replace(".", "p")
    return f"nd{noise_dim}_ns{ns}_{noise_type}"

NOISE_TAG = noise_tag(NOISE_DIM, NOISE_SCALE, NOISE_TYPE)
SEED_TAG = f"seed_{SEED}"

print("DEVICE:", DEVICE)
print("ENV_NAME:", ENV_NAME)
print("NOISE_TAG:", NOISE_TAG)
print("Lambda values:", LAMBDA_VALUES)

In [ ]:
dataset = NoisyOfflineRLDataset(
    env_name=ENV_NAME,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    seed=SEED,
    use_timeouts=True,
    noise_type=NOISE_TYPE,
)

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)

state_dim = dataset.noisy_obs.shape[1]
action_dim = dataset.actions.shape[1]
true_state_dim = dataset.obs_dim
LATENT_DIM = int(max(true_state_dim, NOISE_DIM) * 1.5)

print("state_dim:", state_dim)
print("true_state_dim:", true_state_dim)
print("action_dim:", action_dim)
print("latent_dim:", LATENT_DIM)

In [ ]:
import json

all_results = []

for method_key in SWEEP_METHODS:
    for lam in LAMBDA_VALUES:
        method_full = f"lambda_sweep_{method_key}_lam{str(lam).replace('.', 'p')}"
        print(f"\n{'='*60}")
        print(f"[sweep] method={method_key}, lambda={lam}")
        print(f"{'='*60}")

        CKPT_DIR = CHECKPOINTS_DIR / method_full / ENV_NAME / NOISE_TAG / SEED_TAG
        METRICS_DIR = RAW_METRICS_DIR / method_full / ENV_NAME / NOISE_TAG / SEED_TAG
        OBS_DIR = OBS_STATS_DIR / method_full / ENV_NAME / NOISE_TAG / SEED_TAG
        for d in [CKPT_DIR, METRICS_DIR, OBS_DIR]:
            d.mkdir(parents=True, exist_ok=True)

        np.savez(
            OBS_DIR / "obs_stats.npz",
            obs_mean=dataset.obs_mean,
            obs_std=dataset.obs_std,
            true_state_dim=true_state_dim,
        )

        torch.manual_seed(SEED)
        np.random.seed(SEED)

        encoder = DisentangledEncoder(
            state_dim=state_dim,
            action_dim=action_dim,
            true_state_dim=true_state_dim,
            latent_dim=LATENT_DIM,
        ).to(DEVICE)

        optimizer = torch.optim.Adam(encoder.parameters(), lr=3e-4)
        pretrain_loader = DataLoader(dataset, batch_size=PRETRAIN_BS, shuffle=True, drop_last=True,
                                     num_workers=4, pin_memory=True, persistent_workers=True)

        for epoch in range(1, PRETRAIN_EPOCHS + 1):
            encoder.train()
            losses = []
            for obs, act, next_obs, rew, done, pure_obs, pure_next_obs in pretrain_loader:
                obs = obs.to(DEVICE)
                act = act.to(DEVICE)
                pure_next_obs = pure_next_obs.to(DEVICE)
                rew = rew.to(DEVICE)

                z_task, z_irrel = encoder(obs)

                pred_next = encoder.state_predictor(torch.cat([z_task, act], dim=-1))
                loss_dyn = torch.nn.functional.mse_loss(pred_next, pure_next_obs)

                pred_rew = encoder.reward_predictor(z_task)
                loss_rew = torch.nn.functional.mse_loss(pred_rew.squeeze(-1), rew)

                if method_key == "hsic":
                    loss_indep = hsic_loss(z_task, z_irrel, sigma=1.0)
                else:  # barlow
                    loss_indep = barlow_cross_correlation_loss(z_task, z_irrel)

                loss = loss_dyn + loss_rew + lam * loss_indep

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
                optimizer.step()
                losses.append(float(loss.item()))

            if epoch % 5 == 0 or epoch == PRETRAIN_EPOCHS:
                print(f"  [pretrain] epoch {epoch}, loss={np.mean(losses):.4f}")

        ckpt_encoder = CKPT_DIR / f"encoder_epoch_{PRETRAIN_EPOCHS}.pth"
        torch.save(encoder.state_dict(), ckpt_encoder)

        encoder.eval()
        for p in encoder.parameters():
            p.requires_grad = False

        iql = IQLAgent(
            latent_dim=LATENT_DIM,
            action_dim=action_dim,
            device=DEVICE,
            expectile=0.7,
            temperature=3.0,
            discount=0.99,
        )

        iql_history = train_iql_from_loader(
            iql=iql,
            train_loader=train_loader,
            device=DEVICE,
            epochs=EPOCHS,
            ckpt_dir=CKPT_DIR,
            method=method_full,
            save_every=EPOCHS,
            encoder=encoder,
            repr_mode="disentangled",
            use_tqdm=False,
        )

        metrics = eval_policy_on_env(
            iql=iql,
            env_name=ENV_NAME,
            encoder=encoder,
            method=method_full,
            obs_mean=dataset.obs_mean,
            obs_std=dataset.obs_std,
            true_state_dim=true_state_dim,
            noise_dim=NOISE_DIM,
            noise_scale=NOISE_SCALE,
            noise_type=NOISE_TYPE,
            episodes=20,
            max_steps=1000,
            seed=SEED,
            device=DEVICE,
            use_fixed_noise=True,
        )

        result = {
            "method_key": method_key,
            "lambda": lam,
            "method_full": method_full,
            **metrics,
        }
        all_results.append(result)

        save_metrics_json(
            metrics_dir=METRICS_DIR,
            metrics=metrics,
            env_name=ENV_NAME,
            method=method_full,
            seed=SEED,
            noise_dim=NOISE_DIM,
            noise_scale=NOISE_SCALE,
            noise_type=NOISE_TYPE,
            extra={"lambda": lam, "method_key": method_key},
        )

        print(f"  [result] lambda={lam}, score={metrics['normalized_score']:.2f}")

print("\n[sweep complete]")
for r in all_results:
    print(f"  {r['method_key']:10s} lambda={r['lambda']:8.3f}  score={r['normalized_score']:.2f}")

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)
print(df[["method_key", "lambda", "normalized_score"]].to_string(index=False))

# Save summary CSV
summary_path = (
    ROOT / "results" / "tables" / "ablations" / ENV_NAME
    / f"lambda_sensitivity_{NOISE_TAG}_seed{SEED}.csv"
)
summary_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(summary_path, index=False)
print("Saved summary:", summary_path)